In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

BuGn9 = ["#f7fcfd","#e5f5f9","#ccece6","#99d8c9","#66c2a4","#41ae76","#238b45","#006d2c","#00441b"]
BuGn_new = ["#f7fcfd","#04702F","#006227","#005321","#00441B"]
BuGn_new_cmap = LinearSegmentedColormap.from_list('custom',BuGn_new)

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

fig_dir = "Reproducibility/Results/Plots/MSC/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
tmp_subset = 'MSC'

adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["proCAF","iCAF_CD321",'iCAF_SLC14A1',"matCAF","matCAP",
                                        "contCAP","vSMC"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

## Hotspot

In [ ]:
from scipy.io import mmread
import scipy.sparse as sp
import hotspot

rna_adata = adata[:, adata.var.modality == "Gene Expression"].copy()
rna_adata = rna_adata[:, rna_adata.var['black_list']=='keep']
rna_adata.layers["counts"] = sp.csc_matrix(rna_adata.X)
sc.pp.highly_variable_genes(
            rna_adata,
            n_top_genes=3000,
            flavor="seurat_v3",
            subset=True
        )

In [ ]:
# Creating the Hotspot object
hs = hotspot.Hotspot(
    rna_adata,
    layer_key="counts",
    model='danb',
    latent_obsm_key="MultiVI_latent",
    umi_counts_obs_key="total_counts_RNA"
)

hs.create_knn_graph(weighted_graph=False, n_neighbors=30)

In [ ]:
# Compute pair-wise local correlations between these genes
hs_results = hs.compute_autocorrelations(jobs=4)
hs_genes = hs_results.loc[hs_results.FDR < 1e-100].index
local_correlations = hs.compute_local_correlations(hs_genes, jobs=16)

In [ ]:
# make modules
modules = hs.create_modules(
    min_gene_threshold=80, core_only=True, fdr_threshold=0.01
)

In [ ]:
for tmp_num in range(len(hs.modules)):
    if hs.modules[tmp_num] == 1:
        hs.modules[tmp_num] = 1
    elif hs.modules[tmp_num] == 3:
        hs.modules[tmp_num] = 1
    elif hs.modules[tmp_num] == 2:
        hs.modules[tmp_num] = 2
    elif hs.modules[tmp_num] == 7:
        hs.modules[tmp_num] = 3    
    elif hs.modules[tmp_num] == 4:
        hs.modules[tmp_num] = 4
    elif hs.modules[tmp_num] == 5:
        hs.modules[tmp_num] = 4  
    elif hs.modules[tmp_num] == 6:
        hs.modules[tmp_num] = 5  
    else:
        hs.modules[tmp_num] = -1    

In [ ]:
inferno_r = plt.cm.get_cmap('inferno_r', 256)
first_color_rgb = inferno_r(0)[:3]
new_colors = np.empty((256, 4))
new_colors[:128] = np.hstack([first_color_rgb, 1])
for i in range(128, 256):
    new_colors[i, :3] = inferno_r((i - 128) / 128)[:3]
    new_colors[i, 3] = 1  # Full opacity

custom_inferno_r = LinearSegmentedColormap.from_list("custom_inferno_r", new_colors)

In [ ]:
# Fig.S4A
fig = plt.figure(figsize=(2, 2))
fig = hs.plot_local_correlations(vmin=-25, vmax=25,z_cmap=custom_inferno_r,mod_cmap="Dark2")
plt.savefig(f"{fig_dir}FigureS4A.png", bbox_inches='tight')
plt.close()

In [ ]:
module_scores = hs.calculate_module_scores()
module_scores.columns = [f"MP{x}" for x in module_scores.columns]

rna_adata_tmp = rna_adata.copy()
rna_adata_tmp.obs = pd.merge(left=rna_adata_tmp.obs, right=module_scores, how='left', left_index=True, right_index=True)

In [ ]:
tmp_path = "Reproducibility/Results/Hotspot/MSC/UC_DOGMA_MSC_module_scores.txt"
module_scores.to_csv(tmp_path, sep='\t')

In [ ]:
dir_path = "Reproducibility/Results/Hotspot/MSC/"
os.makedirs(dir_path, exist_ok = True)

for module in np.sort(modules.unique()[modules.unique() >= 0]):
    gl = list(modules[modules == module].index.values)
    file = f"{dir_path}UC_DOGMA_module_" + str(module) + "_gene_list.txt"
    s = "\n".join(gl)
    with open(file, "w", encoding="utf-8") as f:
      f.write(s)

## Visualization

### UMAP

In [ ]:
# Fig.3A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}Figure3A.pdf", bbox_inches='tight')
plt.close()

### RNA dotplot

In [ ]:
adata_RNA = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

In [ ]:
marker_genes = ["LUM","PDGFRA","CFD","C3","C7",
                "ITGB8","FGF10","GREM2","F11R",
                "NRG1","TRPA1","CXCL14",'SLC14A1','ITGA11',"PDPN",
                "MMP11","POSTN",'FAP',"LRRC15", 
                "CDH13","ASPN","DGKI",
                "THY1","PDGFA",
                "RGS5","ACTA2","MYH11",'TAGLN'
                ]  

In [ ]:
# Fig.3C
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby= 'celltype', 
              dendrogram = False,
              standard_scale="var",
              show = False)
plt.savefig(f"{fig_dir}Figure3C.pdf", bbox_inches='tight')
plt.close()

### Protein featureplot

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from palettable.colorbrewer.sequential import YlGn_9

# Create a new colormap by interpolating the original colormap
def rescale_colormap(colormap, rescale_values):
    """
    Rescales a colormap to emphasize specific ranges.
    """
    original_colors = [colormap(v) for v in np.linspace(0, 1, len(rescale_values))]
    new_colormap = LinearSegmentedColormap.from_list("rescaled_colormap", list(zip(rescale_values, original_colors)))
    return new_colormap

rescale_values = [0.0,0.50,0.60,0.70,0.80,0.90,1.0]  # Ensure it starts from 0.0
rescaled_YlGn = rescale_colormap(YlGn_9.mpl_colormap, rescale_values)

In [ ]:
# Fig.3D
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    pro_adata,
    color=["PDGFRa","Podoplanin","CD34","CD321","CD10","CD146","Thy1"],
    color_map = rescaled_YlGn,  
    ncols=4,
    vmax="p99",
    vmin="p10",
    use_raw=False, 
    frameon=False,
    size = 25,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"{fig_dir}Figure3D.pdf", bbox_inches='tight')
plt.close()

### Progeny heatmap

In [ ]:
import decoupler as dc

progeny = dc.get_progeny(organism='human', top=500)
dc.run_mlm(
    mat=adata_RNA,
    net=progeny,
    source='source',
    target='target',
    weight='weight',
    use_raw=False,
    verbose=True
)

In [ ]:
acts = dc.get_acts(adata_RNA, obsm_key='mlm_estimate')
sc.pp.scale(acts)

acts.obs["celltype"] = adata_RNA.obs["celltype"]

In [ ]:
# Fig.3F
import palettable

sc.pl.matrixplot(acts, 
                 var_names=['NFkB','WNT','Hypoxia','TGFb','TNFa','VEGF','JAK-STAT','MAPK','PI3K'],
                 groupby='celltype', 
                 dendrogram=False,
                 colorbar_title='Z-scaled scores', 
                 vmin=-1.5, vmax=1.5, 
                 cmap=palettable.cmocean.diverging.Curl_9.mpl_colormap,
                 show = False)
plt.savefig(f"{fig_dir}Figure3F.pdf", bbox_inches='tight')
plt.close()

### Signature score

In [ ]:
tmp_path1 = 'Reproducibility/Results/VISIONR/UC_DOGMA_MSC_signature_score_hotspot.txt'
sig_df_1 = pd.read_csv(tmp_path1, index_col=0, sep='\t')

tmp_path2 = 'Reproducibility/Results/VISIONR/UC_DOGMA_MSC_signature_score_literature.txt'
sig_df_2 = pd.read_csv(tmp_path2, index_col=0, sep='\t')

sig_df = pd.concat([sig_df_1, sig_df_2], axis=1)

adata.obs = pd.merge(left=adata.obs, right=sig_df.loc[adata.obs_names,:], how='left', left_index=True, right_index=True)

In [ ]:
sc.set_figure_params(figsize=(2, 2)) 
sc.set_figure_params(vector_friendly=True, dpi_save=150)
sc.pl.umap(
    adata,
    color=['Progenitor','Extracellular_matrix','C3_irCAF', 
           'Detox-iCAF','Fibro-matrix'],
    color_map = "magma",    
    ncols=3,
    vmax="p99",
    vmin = 'p1',
#    use_raw=True,  ## raw data
    frameon=False,
    size = 20,
    wspace=0.5,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"{fig_dir}FigureS4C.pdf", bbox_inches='tight')
plt.close()

### TF activity

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_MSC.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

adata_TF = sc.AnnData(TF_activity.T.loc[adata.obs_names,:].copy(), obs=adata.obs)
sc.pp.scale(adata_TF, max_value=10)

adata_TF.obsm['X_umap'] = adata.obsm['X_umap']

In [ ]:
sc.set_figure_params(figsize=(3, 3)) 
sc.set_figure_params(vector_friendly=True, dpi_save=200)
sc.pl.umap(
    adata_TF,
    color=['CEBPD','PITX1','RUNX1','TWIST1','EBF1'],
    color_map = solar_extra,
    ncols=3,
    vmax = "p99",
    vcenter = 0,
    frameon=False,
    wspace=0.1,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"{fig_dir}FigureS4E.pdf", bbox_inches='tight')
plt.close()